In [1]:
%matplotlib qt
import matplotlib.pyplot as plt
import mne
import numpy as np
import os

In [2]:
def drop_epochs_incorrect_buttonpress_resp(events):
    """
    Function to drop frequent events (code: 4) with >0 buttonpress response (code: 32),
    and rare events (code: 5) with 0 or >1 buttonpress responses.
    """
    # Get list of events in order
    event_ord = events[:, 2]
    event_ord = np.append(event_ord, [0, 0])    # add 2 fake event codes for running loop near boundary

    # Find freq events with >0 buttonpresses
    inds_freq_drop = [i_ev-1 for i_ev, ev in enumerate(event_ord) if (ev == 32) and (event_ord[i_ev-1] == 4)]

    # Find rare events with >1 or 0 buttonpresses
    inds_rare_drop = [i_ev for i_ev, ev in enumerate(event_ord) if (ev == 5) and (event_ord[i_ev+1] != 32)] + \
        [i_ev for i_ev, ev in enumerate(event_ord) if (ev == 5) and (event_ord[i_ev+1] == 32) and (event_ord[i_ev+2] == 32)]

    # Drop epochs with incorrect behav resps from events array
    events_filt = np.delete(events, inds_freq_drop + inds_rare_drop, axis=0)

    # Drop buttonpress events
    events_filt = events_filt[(events_filt[:, 2] == 4) | (events_filt[:, 2] == 5)]

    return events_filt

In [3]:
## General settings
plot_all_graphs = False

## Read sample ERP data
# Load raw data
data_dir = mne.datasets.sample.data_path() / "MEG" / "sample"
fpath_raw = data_dir / "sample_audvis_raw.fif"
raw = mne.io.read_raw_fif(fpath_raw, preload=False)

# Load events
fpath_events = data_dir / "sample_audvis_raw-eve.fif"
events = mne.read_events(fpath_events)

# Edit events to use for this analysis
events = events[(events[:, 2] == 4) | (events[:, 2] == 5) | (events[:, 2] == 32)]   # drop unnecc epochs
event_id = {
    "freq": 4,
    "rare": 5,
    # "buttonpress": 32,
}   # event dict for desired epochs

# Pick only EEG and EOG data
raw.pick(["eeg", "eog"]).load_data()

# Remove avg reference proj
raw.del_proj()
display(raw.info)

# Visualize channel locations
# raw.plot_sensors(show_names=True)

Opening raw data file /Users/chholakp2/mne_data/MNE-sample-data/MEG/sample/sample_audvis_raw.fif...
    Read a total of 3 projection items:
        PCA-v1 (1 x 102)  idle
        PCA-v2 (1 x 102)  idle
        PCA-v3 (1 x 102)  idle
    Range : 25800 ... 192599 =     42.956 ...   320.670 secs
Ready.
Removing projector <Projection | PCA-v1, active : False, n_channels : 102>
Removing projector <Projection | PCA-v2, active : False, n_channels : 102>
Removing projector <Projection | PCA-v3, active : False, n_channels : 102>
Reading 0 ... 166799  =      0.000 ...   277.714 secs...


Measurement date,"December 03, 2002 19:01:10 GMT"
Experimenter,MEG
Participant,Unknown
Digitized points,146 points
Good channels,"59 EEG, 1 EOG"
Bad channels,EEG 053
EOG channels,EOG 061
ECG channels,Not available
Sampling frequency,600.61 Hz
Highpass,0.10 Hz
Lowpass,172.18 Hz


### Pfefferbaum et al. 1991

Channel(s): Fz, Pz, and Cz (channels 12, 30, and 48, respectively in MNE-Sample data)

Filtering: 0.13-70 Hz (down 3 dB)

Fs: 200 Hz

Baseline: 100 ms

Epoch: 1150 ms

- Exclude trials with incorrect behavioral response, electrical artifacts (EEG >+- 200 μV), V/HEOG artifacts (>+- 100 μV).
- Rare stim trials averaged separately to give conventional ERP.
- P3 measured between 275-600 ms.

**NOTES**
- The frequency at which the power level of the signal decreases by 3 dB from its maximum value is called the 3 dB bandwidth. A 3 dB decrease in power means the signal power becomes half of its maximum value. (for details, see https://www.everythingrf.com/community/what-is-3-db-bandwidth-in-a-filter)

**NOT POSSIBLE**:
- sternovertebral reference

_______________________________________________________

__Measurement(s)__

P300 amp at Fz, Cz, and Pz; -- [3]

P300 lat at Pz; -- [1]

__Total Measurement(s)__ = 3 + 1 = 4.

__Total feature(s)__ = __Total measurement(s)__ = 4.

In [15]:
def p300_M1(raw, events, event_id, plot_all_graphs=False):
    """
    Calculate P300 chars using the method described in Pfefferbaum et al. 1991.
    """

    # Create a copy of raw and events data to use for this method
    raw_method = raw.copy()
    events_method = events.copy()

    # Rename Fz, Cz, Pz, and EOG and drop data from other channels
    _ = raw_method.rename_channels({"EEG 012": "Fz", "EEG 030": "Cz", "EEG 048": "Pz", "EOG 061": "EOG"}) # happens in-place
    raw_method.pick(["Fz", "Cz", "Pz", "EOG"]);

    # Drop events with incorrect behavioral resps
    events_method = drop_epochs_incorrect_buttonpress_resp(events_method)

    # Lowpass filter raw data from 0.13-70 Hz
    raw_method_filtered = raw_method.copy().filter(l_freq=0.13, h_freq=70)
    if plot_all_graphs:
        raw_method_filtered.compute_psd(fmax=250).plot(average=True, exclude="bads")

    # Decimate (resample) data when epoching; Fs_desired = 200 Hz; [-100, 1150] ms
    desired_sfreq = 200 # Hz
    decim = np.round(raw_method.info["sfreq"] / desired_sfreq).astype(int)
    epochs_method = mne.Epochs(
        raw_method_filtered,
        events_method,  # load from filtered (free from incorr behav resp) events array
        event_id=event_id,
        tmin=-0.1,  # set baseline time
        tmax=1.15,  # set epoch time
        baseline=(None, 0),
        preload=False,
        proj=False,
        decim=decim # resample at 200 Hz
        )

    # Find epochs to be rejected based on eeg amp
    eeg_thresh = 200e-6; eog_thresh = 100e-6
    i_reject_eeg = [i_ep for i_ep, ep in enumerate(epochs_method.get_data(['eeg'])) if np.abs(ep).max() > eeg_thresh]
    i_reject_eog = [i_ep for i_ep, ep in enumerate(epochs_method.get_data(['eog'])) if np.abs(ep).max() > eog_thresh]

    # Drop epochs with EEG/EOG artifacts
    epochs_method.drop(i_reject_eeg, reason="EEGAboveThreshold")
    epochs_method.drop(i_reject_eog, reason="EOGAboveThreshold")
    if plot_all_graphs:
        epochs_method.plot_drop_log()

    # Average rare/frequent epochs to get ERP
    erp_rare = epochs_method["rare"].average()
    # erp_freq = epochs_method["freq"].average()
    if plot_all_graphs:
        erp_rare.plot(spatial_colors=True)
        # erp_freq.plot(spatial_colors=True)
        # mne.viz.plot_compare_evokeds({"rare": erp_rare, "freq": erp_freq}, combine="mean")

    # Measure P3 amps at all 3 sites at Pz P3 latency
    #   Measure P3 amp and lat at Pz
    _, lat_inx, _ = erp_rare.copy().pick(["Pz"]).get_peak(
        tmin=0.275, tmax=0.6, mode="pos", time_as_index=True, return_amplitude=True
    )
    amps = {ch: amp for ch, amp in zip(erp_rare.ch_names, erp_rare.get_data(['eeg'])[:, lat_inx])}
    lat = erp_rare.times[lat_inx]
    return *amps.values(), lat

In [21]:
Fj_M1 = p300_M1(raw, events, event_id=event_id, plot_all_graphs=False)  # returns amps (Fz, Cz, Pz) and lat
print("\n\nF1_M1 = %0.3e." % Fj_M1[0])

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.13 - 70 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.13
- Lower transition bandwidth: 0.13 Hz (-6 dB cutoff frequency: 0.07 Hz)
- Upper passband edge: 70.00 Hz
- Upper transition bandwidth: 17.50 Hz (-6 dB cutoff frequency: 78.75 Hz)
- Filter length: 15247 samples (25.386 s)



Not setting metadata
85 matching events found
Setting baseline interval to [-0.09989760657919393, 0.0] s
Applying baseline correction (mode: mean)
Using data from preloaded Raw for 85 events and 752 original time points (prior to decimation) ...
0 bad epochs dropped
Using data from preloaded Raw for 85 events and 752 original time points (prior to decimation) ...
Dropped 0 epochs: 
Dropped 21 epochs: 3, 9, 17, 21, 26, 31, 35, 38, 46, 49, 51, 52, 53, 54, 55, 57, 60, 68, 73, 75, 84


F1_M1 = 3.206e-06.


/var/folders/8s/6ch1361n4dz5r08lps2_3d_n34sk04/T/ipykernel_46702/3410553913.py:25: RuntimeWarning: The measurement information indicates a low-pass frequency of 70.0 Hz. The decim=3 parameter will result in a sampling frequency of 200.20499674479166 Hz, which can cause aliasing artifacts.
  epochs_method = mne.Epochs(


In [ ]:
# # Create a copy of raw and events data to use for this method
# raw_method = raw.copy()
# events_method = events.copy()

# # Rename Fz, Cz, Pz, and EOG and drop data from other channels
# _ = raw_method.rename_channels({"EEG 012": "Fz", "EEG 030": "Cz", "EEG 048": "Pz", "EOG 061": "EOG"}) # happens in-place
# raw_method.pick(["Fz", "Cz", "Pz", "EOG"]);

# # Drop events with incorrect behavioral resps
# events_method = drop_epochs_incorrect_buttonpress_resp(events_method)

# # Lowpass filter raw data from 0.13-70 Hz
# raw_method_filtered = raw_method.copy().filter(l_freq=0.13, h_freq=70)
# if plot_all_graphs:
#     raw_method_filtered.compute_psd(fmax=250).plot(average=True, exclude="bads")

# # Decimate (resample) data when epoching; Fs_desired = 200 Hz; [-100, 1150] ms
# desired_sfreq = 200 # Hz
# decim = np.round(raw_method.info["sfreq"] / desired_sfreq).astype(int)
# epochs_method = mne.Epochs(
#     raw_method_filtered,
#     events_method,  # load from filtered (free from incorr behav resp) events array
#     event_id=event_id,
#     tmin=-0.1,  # set baseline time
#     tmax=1.15,  # set epoch time
#     baseline=(None, 0),
#     preload=False,
#     proj=False,
#     decim=decim # resample at 200 Hz
#     )

# # Find epochs to be rejected based on eeg amp
# eeg_thresh = 200e-6; eog_thresh = 100e-6
# i_reject_eeg = [i_ep for i_ep, ep in enumerate(epochs_method.get_data(['eeg'])) if np.abs(ep).max() > eeg_thresh]
# i_reject_eog = [i_ep for i_ep, ep in enumerate(epochs_method.get_data(['eog'])) if np.abs(ep).max() > eog_thresh]

# # Drop epochs with EEG/EOG artifacts
# epochs_method.drop(i_reject_eeg, reason="EEGAboveThreshold")
# epochs_method.drop(i_reject_eog, reason="EOGAboveThreshold")
# if plot_all_graphs:
#     epochs_method.plot_drop_log()

# # Average rare/frequent epochs to get ERP
# erp_rare = epochs_method["rare"].average()
# erp_freq = epochs_method["freq"].average()
# if plot_all_graphs:
#     erp_rare.plot(spatial_colors=True)
#     erp_freq.plot(spatial_colors=True)
#     mne.viz.plot_compare_evokeds({"rare": erp_rare, "freq": erp_freq}, combine="mean")

# # Measure P3 amps at all 3 sites at Pz P3 latency
# #   Measure P3 amp and lat at Pz
# _, lat_inx, _ = erp_rare.copy().pick(["Pz"]).get_peak(
#     tmin=0.275, tmax=0.6, mode="pos", time_as_index=True, return_amplitude=True
# )
# amps = {ch: amp for ch, amp in zip(erp_rare.ch_names, erp_rare.get_data(['eeg'])[:, lat_inx])}
# lat = erp_rare.times[lat_inx]
# display(amps, lat)